In [ ]:
%matplotlib inline
# =============================================================================
# 全卷积网络（FCN）语义分割实现
# =============================================================================
# 本模块实现FCN（Fully Convolutional Network）用于语义分割任务
#
# FCN核心思想：
# 1. 使用预训练的CNN（如ResNet）作为骨干网络提取特征
# 2. 将全连接层替换为卷积层（1x1卷积）
# 3. 使用转置卷积（反卷积）进行上采样，恢复空间分辨率
# 4. 输出与输入等大的分割图，每个像素预测一个类别
#
# 关键概念：
# - 编码器（Encoder）：逐层下采样提取语义特征
# - 解码器（Decoder）：逐层上采样恢复空间细节
# - 转置卷积：可学习的上采样操作
# =============================================================================

import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l


# =============================================================================
# 1. 加载预训练模型
# =============================================================================
# 使用预训练的ResNet-18作为骨干网络
# ResNet在ImageNet上预训练，已经学到丰富的图像特征
pretrained_net = torchvision.models.resnet18(pretrained=True)

# 查看ResNet的最后几层结构
# - [-3]: layer4（最后一个残差块，输出通道512）
# - [-2]: avgpool（全局平均池化）
# - [-1]: fc（全连接层，用于分类）
print("ResNet最后3层结构:")
print(list(pretrained_net.children())[-3:])

# =============================================================================
# 2. 构建FCN编码器（去除全连接层）
# =============================================================================
# 去掉最后的全局平均池化和全连接层
# 保留卷积部分作为特征提取器
# 输入: (N, 3, H, W)
# 输出: (N, 512, H/32, W/32) - 空间分辨率降低32倍
net = nn.Sequential(*list(pretrained_net.children())[:-2])

# 测试网络输出形状
X = torch.rand(size=(1, 3, 320, 480))
print(f"\n输入形状: {X.shape}")
print(f"ResNet编码器输出形状: {net(X).shape}")
print("说明: 空间分辨率从(320,480)降为(10,15)，通道数变为512")


# =============================================================================
# 3. 添加FCN解码器
# =============================================================================
# FCN解码器包含两部分：
# 1. 1x1卷积：将通道数从512映射到类别数（21类，VOC数据集）
# 2. 转置卷积：上采样32倍，恢复原始分辨率

num_classes = 21  # VOC数据集有20个前景类 + 1个背景类

# 添加1x1卷积层（替换原来的全连接层）
# 作用：将通道数从512降为类别数，保持空间分辨率
net.add_module('final_conv', nn.Conv2d(512, num_classes, kernel_size=1))

# 添加转置卷积层（可学习的上采样）
# 参数说明：
#   - kernel_size=64: 大卷积核确保平滑的上采样
#   - stride=32: 上采样32倍
#   - padding=16: 保持输出尺寸 = 输入尺寸 * 32
net.add_module('transpose_conv', nn.ConvTranspose2d(
    num_classes, num_classes, kernel_size=64, padding=16, stride=32))

print(f"\n完整FCN网络结构（最后两层）:")
print(f"final_conv: Conv2d(512, {num_classes}, kernel_size=1)")
print(f"transpose_conv: ConvTranspose2d({num_classes}, {num_classes}, kernel_size=64, stride=32)")


# =============================================================================
# 4. 双线性插值初始化
# =============================================================================
# 转置卷积使用双线性插值核初始化，确保初始状态就是平滑上采样
# 这样训练更容易收敛

def bilinear_kernel(in_channels, out_channels, kernel_size):
    """
    生成双线性插值核
    
    原理：
    双线性插值是一种图像缩放算法，通过加权平均相邻像素计算新像素值
    这个函数生成一个卷积核，使得转置卷积执行双线性上采样
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 卷积核大小（通常是stride的倍数）
    
    返回:
        weight: 初始化用的卷积核权重
    """
    factor = (kernel_size + 1) // 2
    if kernel_size % 2 == 1:
        center = factor - 1
    else:
        center = factor - 0.5
    
    # 生成二维网格坐标
    og = (torch.arange(kernel_size).reshape(-1, 1),
          torch.arange(kernel_size).reshape(1, -1))
    
    # 计算双线性插值权重
    # 距离中心越近，权重越大
    filt = (1 - torch.abs(og[0] - center) / factor) * \
           (1 - torch.abs(og[1] - center) / factor)
    
    # 为每个输入/输出通道对复制权重
    weight = torch.zeros((in_channels, out_channels,
                          kernel_size, kernel_size))
    weight[range(in_channels), range(out_channels), :, :] = filt
    
    return weight


# 测试双线性插值上采样效果
conv_trans = nn.ConvTranspose2d(3, 3, kernel_size=4, padding=1, stride=2,
                                bias=False)
conv_trans.weight.data.copy_(bilinear_kernel(3, 3, 4))

# 加载测试图片
img = torchvision.transforms.ToTensor()(d2l.Image.open('../img/catdog.jpg'))
X = img.unsqueeze(0)  # 添加batch维度
Y = conv_trans(X)     # 上采样2倍
out_img = Y[0].permute(1, 2, 0).detach()

print(f"\n双线性插值上采样测试:")
print(f"输入图片形状: {img.shape}")
print(f"输出图片形状: {out_img.shape}")
print(f"上采样倍数: 2x")

# 使用双线性核初始化FCN的转置卷积层
W = bilinear_kernel(num_classes, num_classes, 64)
net.transpose_conv.weight.data.copy_(W)
print(f"\n转置卷积层已使用双线性核初始化")


# =============================================================================
# 5. 加载数据集
# =============================================================================
batch_size, crop_size = 32, (320, 480)

# 加载VOC语义分割数据集
# 每个样本包含：图片(RGB)和像素级标签（每个像素一个类别）
train_iter, test_iter = d2l.load_data_voc(batch_size, crop_size)

print(f"\n数据集信息:")
print(f"批大小: {batch_size}")
print(f"裁剪尺寸: {crop_size}")


# =============================================================================
# 6. 定义损失函数
# =============================================================================
def loss(inputs, targets):
    """
    语义分割损失函数
    
    使用交叉熵损失，对每个像素的预测计算损失，然后求平均
    
    参数:
        inputs: 模型输出，形状 (batch_size, num_classes, H, W)
        targets: 标签，形状 (batch_size, H, W)，每个像素是类别索引
    
    返回:
        loss: 每个样本的平均损失，形状 (batch_size,)
    """
    # F.cross_entropy: (N, C, H, W) + (N, H, W) -> (N, H, W)
    # reduction='none': 不自动求和或平均，保留每个像素的损失
    # mean(1).mean(1): 对高和宽分别求平均，得到每个样本的损失
    return F.cross_entropy(inputs, targets, reduction='none').mean(1).mean(1)


# =============================================================================
# 7. 训练模型
# =============================================================================
num_epochs, lr, wd, devices = 5, 0.001, 1e-3, d2l.try_all_gpus()

# 使用SGD优化器（带动量）
trainer = torch.optim.SGD(net.parameters(), lr=lr, weight_decay=wd, momentum=0.9)

print(f"\n训练配置:")
print(f"轮数: {num_epochs}, 学习率: {lr}, 权重衰减: {wd}")
print(f"设备: {devices}")

# 开始训练
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs, devices)


# =============================================================================
# 8. 预测与可视化
# =============================================================================
def predict(img):
    """
    对单张图片进行语义分割预测
    
    参数:
        img: 输入图片，形状 (3, H, W)
    
    返回:
        pred: 预测结果，形状 (H, W)，每个像素是类别索引
    """
    # 标准化并添加batch维度
    X = test_iter.dataset.normalize_image(img).unsqueeze(0)
    
    # 前向传播
    # net输出形状: (1, num_classes, H, W)
    # argmax(dim=1): 在类别维度取最大值索引 -> (1, H, W)
    pred = net(X.to(devices[0])).argmax(dim=1)
    
    # 去掉batch维度
    return pred.reshape(pred.shape[1], pred.shape[2])


def label2image(pred):
    """
    将类别索引转换为彩色图像
    
    VOC数据集定义了每个类别的颜色（例如：人是粉色，车是蓝色）
    
    参数:
        pred: 类别索引，形状 (H, W)
    
    返回:
        彩色图像，形状 (H, W, 3)
    """
    colormap = torch.tensor(d2l.VOC_COLORMAP, device=devices[0])
    X = pred.long()
    return colormap[X, :]


# 下载VOC数据集（如果不存在）
voc_dir = d2l.download_extract('voc2012', 'VOCdevkit/VOC2012')
test_images, test_labels = d2l.read_voc_images(voc_dir, False)

# 可视化预测结果
n, imgs = 4, []
for i in range(n):
    crop_rect = (0, 0, 320, 480)
    
    # 裁剪图片
    X = torchvision.transforms.functional.crop(test_images[i], *crop_rect)
    
    # 预测并转换为彩色图
    pred = label2image(predict(X))
    
    # 裁剪真实标签
    label = torchvision.transforms.functional.crop(
        test_labels[i], *crop_rect)
    
    # 收集图片：原图、预测结果、真实标签
    imgs += [X.permute(1, 2, 0), pred.cpu(), label.permute(1, 2, 0)]

# 显示结果（3行：第1行原图，第2行预测，第3行真实标签）
d2l.show_images(imgs[::3] + imgs[1::3] + imgs[2::3], 3, n, scale=2)

print("\n" + "="*60)
print("可视化说明:")
print("第1行: 输入原图")
print("第2行: FCN预测结果")
print("第3行: 真实标签")
print("="*60)